# #2 褒め言葉変換基盤：一括の実測（Colab＋Ollama）

「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選んでから、上から順に実行してください。
結果は最後のセルで `results.zip` にまとめてダウンロードします。

| 実験 | 動画の場所 | 測るもの |
|---|---|---|
| A | 0:51 | スキル相当のプロンプトで同じ指摘を100回変換 → 冷たい言い方の回数 |
| B | 1:58 | 判定LLMの精度（工藤さんがラベルを付けた評価データ） |
| C | 2:44 | 負荷試験：褒め率、変換 p99、重複率 |
| D | 2:35 | カナリア：褒め率の低下、バーンレート、自動ロールバックまでの秒数 |

In [ ]:
# GPUの確認と、Ollamaの導入（数分かかります）
!nvidia-smi --query-gpu=name,memory.total --format=csv
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, os, time
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
MODEL = "qwen2.5:7b"
!ollama pull {MODEL}

In [ ]:
# プロンプト（mac/prompts.py と同じ内容）
CONVERT_V12 = """あなたはコードレビューのコメントを書き換えるアシスタントです。
入力されたレビューコメントを、受け取った人が前向きな気持ちになる褒め言葉に書き換えてください。

ルール：
- 必ず相手や成果物を褒める表現にする
- 否定・命令・問題の指摘に聞こえる言葉は使わない
- 1〜2文、80文字以内
- 書き換えた文だけを出力する（説明や前置きは書かない）"""

CONVERT_V13 = """あなたはコードレビューのコメントを書き換えるアシスタントです。
入力されたレビューコメントを、簡潔に要点だけ伝える文に書き換えてください。

ルール：
- 簡潔に要点だけ伝える
- 1文、40文字以内
- 書き換えた文だけを出力する（説明や前置きは書かない）"""

SOFTEN_SKILL = """雑に書いたレビュー指摘を、やわらかい言い方に変換してください。

変換ルール：
- 指摘の内容は変えない
- 相手を責める表現を使わない
- 前向きな言葉で言い換える
- 1〜2文。変換した文だけを出力する"""

JUDGE = """あなたはコードレビューのコメントを分類する判定器です。
次のコメントが「受け取った人を褒める表現」になっているかを判定してください。

- 褒め言葉になっている → praise: true
- 指摘・命令・否定・冷たい言い方が含まれる → praise: false

JSONだけを出力してください。形式：{"praise": true または false}"""

In [ ]:
import requests, json, time, random, statistics, os
from concurrent.futures import ThreadPoolExecutor
os.makedirs("/content/results", exist_ok=True)

def chat(system, text, temperature=0.7, json_mode=False):
    body = {"model": MODEL, "stream": False, "options": {"temperature": temperature},
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": text}]}
    if json_mode: body["format"] = "json"
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=120)
    return r.json()["message"]["content"].strip()

def judge(text):
    try: return bool(json.loads(chat(JUDGE, text, temperature=0, json_mode=True))["praise"])
    except Exception: return False   # 判定できないものは「褒めていない」扱い

def convert_and_judge(text, prompt=CONVERT_V12, max_attempts=3):
    t0 = time.time(); tries = []
    for _ in range(max_attempts):
        out = chat(prompt, text); ok = judge(out); tries.append({"out": out, "praise": ok})
        if ok: break
    return {"input": text, "final": tries[-1]["out"], "praise": tries[-1]["praise"],
            "attempts": len(tries), "sec": time.time() - t0, "tries": tries}

def pct(xs, q):
    s = sorted(xs); return s[min(len(s)-1, int(round(q*(len(s)-1))))]

def save(name, obj):
    json.dump(obj, open(f"/content/results/{name}.json", "w"), ensure_ascii=False, indent=1)

print(convert_and_judge("ここバグってる"))

## 実験A：スキル相当のプロンプトで、同じ指摘を100回変換（0:51）

In [ ]:
N = 100
outs = []
for i in range(N):
    out = chat(SOFTEN_SKILL, "ここバグってる", temperature=1.0)
    outs.append({"i": i, "out": out, "praise": judge(out)})
cold = [o for o in outs if not o["praise"]]
print(f"冷たい言い方（褒めていない）：{len(cold)} / {N} 回")
for o in cold[:10]: print(" ", o["i"], o["out"])
save("A_variance", {"n": N, "cold": len(cold), "outs": outs})

## 実験B：判定LLMの精度（1:58）

`data/eval_candidates.csv` を開き、**label 列に praise か cold を工藤さんが付けてから**アップロードしてください。
（label_draft は Claude の下書きです。そのまま使わず、必ず自分の判断で付けてください）

In [ ]:
from google.colab import files
import csv, io
up = files.upload()
rows = list(csv.DictReader(io.StringIO(list(up.values())[0].decode("utf-8"))))
rows = [r for r in rows if r["label"].strip() in ("praise", "cold")]
res = []
for r in rows:
    p = judge(r["text"]); res.append({**r, "pred": "praise" if p else "cold"})
correct = sum(r["pred"] == r["label"] for r in res)
missed = [r for r in res if r["label"] == "cold" and r["pred"] == "praise"]   # 冷たいのに「褒め」と判定
false_alarm = [r for r in res if r["label"] == "praise" and r["pred"] == "cold"]
print(f"評価データ：{len(res)}件  正解率：{correct/len(res)*100:.1f}%")
print(f"冷たいのに「褒め」と判定：{len(missed)}件 / 褒めなのに「冷たい」と判定：{len(false_alarm)}件")
for r in missed: print("  見逃し：", r["text"])
save("B_eval", {"n": len(res), "accuracy": correct/len(res), "missed": missed, "false_alarm": false_alarm, "rows": res})

## 実験C：負荷試験（2:44）

過去のコメントの代わりに、テンプレートから作った合成データを使います（動画では「合成データ」と明記）。

In [ ]:
TARGETS = ["ここ","この関数","この変数名","このループ","このSQL","この例外処理","このif文","このテスト","このコメント","このクラス",
           "この定数","このAPI呼び出し","このマジックナンバー","このnullチェック","このログ出力","この正規表現","この依存関係","このタイムアウト値","このキャッシュ","このリトライ処理"]
PROBLEMS = ["バグってる","意味がわからない","重複してる","遅い","読みにくい","名前が微妙","テストがない","不要では","責務が多すぎる","エラーを握りつぶしてる",
            "型が違う","境界値が漏れてる","N+1になってる","競合しそう","メモリリークしそう","命名規則と違う","ハードコードされてる","副作用がある","戻り値を見てない","ネストが深い",
            "古い書き方","セキュリティ的に危ない","ログが多すぎる","コメントと実装が違う","直して"]
SUFFIX = ["", "。", "。直して", "？"]
POOL = [f"{t}、{p}{s}" for t in TARGETS for p in PROBLEMS for s in SUFFIX]
random.seed(42)

N = 1000          # GPUの速さを見て調整（まず100で試すのがおすすめ）
CONCURRENCY = 4
comments = random.sample(POOL, N)
t0 = time.time()
with ThreadPoolExecutor(CONCURRENCY) as ex:
    load = list(ex.map(convert_and_judge, comments))
wall = time.time() - t0
praise_rate = sum(r["praise"] for r in load) / N
secs = [r["sec"] for r in load]
finals = [r["final"] for r in load]
dup_rate = 1 - len(set(finals)) / N
print(f"件数：{N}  所要：{wall/60:.1f}分")
print(f"褒め率：{praise_rate*100:.2f}%  （褒めていないまま届いた：{N - sum(r['praise'] for r in load)}件）")
print(f"変換（判定と再変換を含む） p50={pct(secs,.5):.2f}s  p99={pct(secs,.99):.2f}s")
print(f"褒め言葉の重複率：{dup_rate*100:.1f}%  再変換が起きた割合：{sum(r['attempts']>1 for r in load)/N*100:.1f}%")
save("C_load", {"n": N, "wall_sec": wall, "praise_rate": praise_rate, "p50": pct(secs,.5), "p99": pct(secs,.99),
                "dup_rate": dup_rate, "results": load})

## 実験D：カナリアリリースと自動ロールバック（2:35）

v12 で流している途中で v13（冷たくなる変更）を 5% に配信し、カナリア側の褒め率からバーンレートを計算します。
SLO 99.99% なのでエラーバジェットは 0.01%。バーンレートが 14.4 倍を超えたら v12 に戻します。

In [ ]:
import threading
SLO = 0.9999
BURN_THRESHOLD = 14.4
CANARY_RATIO = 0.05
WINDOW_SEC = 30          # バーンレートを見る窓
MIN_SAMPLES = 5          # 窓の中にカナリアの結果がこの件数以上あるときだけ判定
WARMUP_SEC = 20          # 最初は v12 だけで流す
MAX_SEC = 300

events = []; lock = threading.Lock()
state = {"canary": False, "rolled_back": False, "t_deploy": None, "t_rollback": None, "max_burn": 0}
t_start = time.time()

def worker(text):
    now = time.time() - t_start
    use_v13 = state["canary"] and not state["rolled_back"] and random.random() < CANARY_RATIO
    out = chat(CONVERT_V13 if use_v13 else CONVERT_V12, text); ok = judge(out)
    with lock:
        events.append({"t": time.time() - t_start, "version": "v13" if use_v13 else "v12", "praise": ok, "out": out})

def monitor():
    while time.time() - t_start < MAX_SEC and not state["rolled_back"]:
        time.sleep(1)
        now = time.time() - t_start
        if not state["canary"] and now >= WARMUP_SEC:
            state["canary"] = True; state["t_deploy"] = now; print(f"{now:6.1f}s  v13 を canary 5% に配信")
        if not state["canary"]: continue
        with lock:
            win = [e for e in events if e["version"] == "v13" and e["t"] >= now - WINDOW_SEC]
        if len(win) < MIN_SAMPLES: continue
        err = 1 - sum(e["praise"] for e in win) / len(win)
        burn = err / (1 - SLO); state["max_burn"] = max(state["max_burn"], burn)
        if burn > BURN_THRESHOLD:
            state["rolled_back"] = True; state["t_rollback"] = now
            print(f"{now:6.1f}s  ALERT バーンレート {burn:.1f}倍 → v12 にロールバック")

mon = threading.Thread(target=monitor); mon.start()
with ThreadPoolExecutor(CONCURRENCY) as ex:
    i = 0
    while mon.is_alive():
        ex.submit(worker, POOL[i % len(POOL)]); i += 1
        time.sleep(0.25)
mon.join()
if state["t_rollback"]:
    print(f"配信からロールバックまで：{state['t_rollback'] - state['t_deploy']:.1f}秒")

# 5秒ごとのカナリア側の褒め率（グラフ用）
series = []
for b in range(0, int(max(e["t"] for e in events)) + 5, 5):
    seg = [e for e in events if b <= e["t"] < b + 5]
    v13 = [e for e in seg if e["version"] == "v13"]
    series.append({"t": b, "all_praise_rate": sum(e["praise"] for e in seg)/len(seg) if seg else None,
                   "canary_n": len(v13), "canary_praise_rate": sum(e["praise"] for e in v13)/len(v13) if v13 else None})
save("D_canary", {**state, "window_sec": WINDOW_SEC, "events": events, "series": series})

## 結果をダウンロード

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download("/content/results.zip")